# 15 Differential Abundance and Settlement

We use ANCOM-BC2 to test whether fungal genus abundance is associated with settlement type (`Settlement_Type`). We use the unrarefied genus count table from notebook 14 and keep the same samples for the unadjusted and adjusted models. The adjusted model accounts for age and diet, as in notebook 08.

## 1. Setup

### 1.1 Imports

In [15]:
import pandas as pd
import qiime2 as q2
from qiime2 import Artifact, Metadata

In [29]:
in_dir = "../data/processed_ignored/14_differential_abundance_setup"
out_dir = "../data/processed_ignored/15_differential_abundance_settlement"
vis_dir = "../data/processed/15_differential_abundance_settlement"
metadata = "../data/processed_ignored/06_diversity/Final_metadata_with_GHSL.tsv"

unadjusted_dir = f"{vis_dir}/unadjusted_ancombc2" 
adjusted_dir = f"{vis_dir}/adjusted_ancombc2"

genus_table = f"{in_dir}/table_fungi_l6.qza"

settlement_table = f"{out_dir}/table_fungi_l6_settlement.qza"

settlement_unadjusted_qza = f"{out_dir}/ancombc2_settlement_unadjusted.qza"
settlement_adjusted_qza = f"{out_dir}/ancombc2_settlement_adjusted.qza"

settlement_unadjusted_tabulate = f"{unadjusted_dir}/tabulate_unadjusted_settlement.qzv"
settlement_adjusted_tabulate = f"{adjusted_dir}/tabulate_adjusted_settlement.qzv"

settlement_unadjusted_barplot = f"{unadjusted_dir}/barplot_unadjusted_settlement.qzv"
settlement_adjusted_barplot = f"{adjusted_dir}/barplot_adjusted_settlement.qzv"

!mkdir -p "$out_dir" "$vis_dir"
!mkdir -p "$unadjusted_dir" "$adjusted_dir"

### 1.3 Load inputs

We load the genus count table from notebook 14 and the cleaned sample metadata used in notebook 08.

In [ ]:
genus_artifact = Artifact.load(genus_table)
sample_metadata = Metadata.load(metadata)

print("Table type:", genus_artifact.type)
print("Samples in metadata:", sample_metadata.id_count)

Table type: FeatureTable[Frequency]
Samples in metadata: 150


## 2. Select samples

### 2.1 Keep complete cases

We keep samples with settlement type, age, and diet information so both models use the same samples.

In [ ]:
!qiime feature-table filter-samples --help

In [8]:
!qiime feature-table filter-samples \
    --i-table "$genus_table" \
    --m-metadata-file "$metadata" \
    --p-where "[Settlement_Type] IS NOT NULL AND [age_years_sample] IS NOT NULL AND [diet_type_sample] IS NOT NULL" \
    --o-filtered-table "$settlement_table"

Saved FeatureTable[Frequency] to: ../data/processed_ignored/15_differential_abundance_settlement/table_fungi_l6_settlement.qza


### 2.2 Summarize selected samples

We count the samples in each settlement group after filtering for complete settlement, age, and diet information.

In [10]:
settlement_counts = Artifact.load(settlement_table).view(pd.DataFrame)
selected_metadata = sample_metadata.to_dataframe().loc[settlement_counts.index]

print("Selected samples:", len(selected_metadata))
print(selected_metadata["Settlement_Type"].value_counts())

Selected samples: 139
Settlement_Type
Rural       61
Urban       54
Suburban    24
Name: count, dtype: int64


In [13]:
!qiime composition ancombc2 --help

Usage: qiime composition ancombc2 [OPTIONS]

  Calls the `ancombc2` function of the ANCOMBC software package. See the
  ANCOM-BC2 publication and source code for details.

  Sensitivity Analysis for Pseudo-Count Addition: To assess robustness, a
  series of pseudo-counts (0.01 to 0.5, in 0.01 increments) is added to zero
  counts, and a linear model is fitted to the bias-corrected log-abundance
  data for each. A sensitivity score is calculated as the proportion of
  p-values above the specified alpha. Taxa with consistent significance across
  all pseudo-counts, and agreement with complete-data results, are flagged as
  robust (diff_robust = TRUE). Note: Results depend on the selected alpha
  value.

Inputs:
  --i-table ARTIFACT FeatureTable[Frequency]
                         The feature table to be used for ANCOM-BC2
                         computation.                               [required]
Parameters:
  --m-metadata-file METADATA...
    (multiple            The per-sample metad

## 3. Run ANCOM-BC2

We use ANCOM-BC2 because some fungal genera were detected in only a few samples. When a genus has many zero counts, a result can depend on the small value used in place of zero. ANCOM-BC2 checks this by trying several small values and seeing whether the result stays significant. 

We add `--p-diff-robust` to both models to see which results stay significant when ANCOM-BC2 tries different small values for the zero counts.

### 3.1 Unadjusted ANCOM-BC2

We test whether fungal genus abundance differs by settlement type. We use Rural as the reference group and compare Suburban and Urban samples with Rural samples.

In [19]:
!qiime composition ancombc2 \
    --i-table "$settlement_table" \
    --m-metadata-file "$metadata" \
    --p-fixed-effects-formula "Settlement_Type" \
    --p-reference-levels "Settlement_Type::Rural" \
    --p-diff-robust \
    --o-ancombc2-output "$settlement_unadjusted_qza"

Saved FeatureData[ANCOMBC2Output] to: ../data/processed_ignored/15_differential_abundance_settlement/ancombc2_settlement_unadjusted.qza


In [22]:
!qiime composition tabulate \
    --i-data "$settlement_unadjusted_qza" \
    --o-visualization "$settlement_unadjusted_tabulate"

!qiime composition da-barplot \
    --i-data "$settlement_unadjusted_qza" \
    --o-visualization "$settlement_unadjusted_barplot"


Saved Visualization to: ../data/processed/15_differential_abundance_settlement/unadjusted_ancombc2/tabulate_unadjusted_settlement.qzv
Saved Visualization to: ../data/processed/15_differential_abundance_settlement/unadjusted_ancombc2/barplot_unadjusted_settlement.qzv


#### Unadjusted results

**Suburban vs. Rural**

| Taxon | Log fold change | q-value | diff_robust |
|---|---:|---:|:---:|
| *Oligophagozyma* | +1.854 | 0.019616 | False |
| *Apiotrichum* | −1.903 | 0.019616 | False |
| *Debaryomyces* | +1.591 | 0.029981 | False |
| *Saccharomyces* | +1.634 | 0.042097 | False |


**Urban vs. Rural**

| Taxon | Log fold change | q-value | diff_robust |
|---|---:|---:|:---:|
| *Naganishia* | +2.154 | 0.000630 | False |
| *Oligophagozyma* | +1.997 | 0.001697 | False |
| *Cladosporium* | −1.330 | 0.032551 | False |
| Dipodascaceae* | −1.433 | 0.044441 | False |
| *Sporormiella* | −1.399 | 0.047224 | False |

*genus unidentified


Four taxa had q < 0.05 for Suburban vs. Rural and five for Urban vs. Rural. *Oligophagozyma* had higher estimated abundance in both groups compared with Rural. None of the nine results passed the sensitivity check for zero counts.

### 3.2 Adjusted ANCOM-BC2

We compare Urban and Suburban samples with Rural samples after accounting for age and diet. We use the same 139 samples as in the unadjusted model.

In [27]:
!qiime composition ancombc2 \
    --i-table "$settlement_table" \
    --m-metadata-file "$metadata" \
    --p-fixed-effects-formula "age_years_sample + diet_type_sample + Settlement_Type" \
    --p-reference-levels "Settlement_Type::Rural" \
    --p-diff-robust \
    --o-ancombc2-output "$settlement_adjusted_qza"

!qiime composition tabulate \
    --i-data "$settlement_adjusted_qza" \
    --o-visualization "$settlement_adjusted_tabulate"

!qiime composition da-barplot \
    --i-data "$settlement_adjusted_qza" \
    --o-visualization "$settlement_adjusted_barplot"


Saved FeatureData[ANCOMBC2Output] to: ../data/processed_ignored/15_differential_abundance_settlement/ancombc2_settlement_adjusted.qza
Saved Visualization to: ../data/processed/15_differential_abundance_settlement/adjusted_ancombc2/tabulate_adjusted_settlement.qzv
Saved Visualization to: ../data/processed/15_differential_abundance_settlement/adjusted_ancombc2/barplot_adjusted_settlement.qzv


#### Adjusted Results 

**Suburban vs. Rural**

| Genus | LFC | q-value | diff_robust |
|---|---:|---:|:---:|
| *Pichia* | −2.109 | 0.000525 | False |
| *Kluyveromyces* | −2.203 | 0.013335 | False |

**Urban vs. Rural**

| Genus | LFC | q-value | diff_robust |
|---|---:|---:|:---:|
| *Apiotrichum* | +3.252 | 0.003135 | False |
| *Oligophagozyma* | +2.100 | 0.021492 | False |
| *Kluyveromyces* | −2.255 | 0.026612 | False |

After accounting for age and diet, Kluyveromyces had lower estimated abundance in both Suburban and Urban samples compared with Rural samples. Pichia was also lower in Suburban samples, while Apiotrichum and Oligophagozyma were higher in Urban samples. All five results had q < 0.05, but none passed the ANCOM-BC2 sensitivity check for how zero counts were handled (diff_robust = False).

## 4. Discussion

Both models used the same 139 samples. After accounting for age and diet, fewer taxa had q < 0.05. Only *Oligophagozyma* met this threshold for the same comparison in both models. Its estimated abundance was higher in Urban than in Rural samples.

Even taxa with very low q-values had `diff_robust = False`. The q-value accounts for testing many taxa, while `diff_robust` checks whether a finding holds when zero counts are handled differently. All findings missed this second criterion. Many fungi occurred in only a few samples, which may help explain why the results were sensitive to zeros. We therefore interpret these settlement differences cautiously.